In [0]:
import dlt
from pyspark.sql.functions import col, window, max, avg, round

@dlt.table(
    name="03_gold.bridge_metrics",
    comment="10-min avg temperature, max vibration & max tilt per bridge with window start/end"
)
def bridge_metrics():

    # Watermarked streams
    temp = (
        dlt.read_stream("02_silver.bridge_temperature")
           .withWatermark("event_time", "2 minutes")
    )

    vib = (
        dlt.read_stream("02_silver.bridge_vibration")
           .withWatermark("event_time", "2 minutes")
    )

    tilt = (
        dlt.read_stream("02_silver.bridge_tilt")
           .withWatermark("event_time", "2 minutes")
    )

    # Windowed aggregations (KEEP window struct)
    temp_agg = (
        temp.groupBy(
                window(col("event_time"), "10 minutes").alias("w"),
                col("bridge_id"),
                col("name"),
                col("location")
            )
            .agg(avg("temperature").alias("avg_temperature"))
    )

    vib_agg = (
        vib.groupBy(
                window(col("event_time"), "10 minutes").alias("w"),
                col("bridge_id")
            )
            .agg(max("vibration").alias("max_vibration"))
    )

    tilt_agg = (
        tilt.groupBy(
                window(col("event_time"), "10 minutes").alias("w"),
                col("bridge_id")
            )
            .agg(max("tilt_angle").alias("max_tilt_angle"))
    )

    # ✅ Stream–stream joins on bridge_id + window struct
    joined = (
        temp_agg
            .join(vib_agg, on=["bridge_id", "w"], how="left")
            .join(tilt_agg, on=["bridge_id", "w"], how="left")
    )

    # Flatten window AFTER join
    return (
        joined.select(
            col("bridge_id"),
            col("name"),
            col("location"),
            col("w.start").alias("window_start"),
            col("w.end").alias("window_end"),
            round(col("avg_temperature"), 2).alias("avg_temperature"),
            round(col("max_vibration"), 4).alias("max_vibration"),
            round(col("max_tilt_angle"), 6).alias("max_tilt_angle")
        )
    )
